<a href="https://colab.research.google.com/github/MoAppOfficial/moapp-colab/blob/main/%D8%A5%D9%86%D8%B4%D8%A7%D8%A1_%D8%AA%D9%88%D8%B1%D9%86%D8%AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === 1. تثبيت الحزم المطلوبة ===
import os
import sys
from IPython.display import clear_output

print("⏳ جاري تثبيت الحزم المطلوبة... يرجى الانتظار.")
os.system('python -m pip install -q -U pip setuptools wheel')
os.system('python -m pip install -q torf')
os.system('python -m pip install -q libtorrent || python -m pip install -q lbry-libtorrent')
clear_output()
print("✅ تم تثبيت الحزم بنجاح!\n")

import json
import glob
import threading
import time
import libtorrent as lt
from torf import Torrent
from google.colab import drive

# ربط جوجل درايف
drive.mount("/content/drive/")

# === إعداد المسارات والذاكرة ===
seed_folder = '/content/Seed_Torrents'
config_file = '/content/drive/MyDrive/torrents_map.json'
os.makedirs(seed_folder, exist_ok=True)

# قائمة التراكرز الخارجية النشطة (تم إضافة التراكر المطلوب بعد opentrackr)
public_trackers = [
    "udp://tracker.opentrackr.org:1337/announce",
    "udp://open.stealth.si:80/announce",
    "udp://open.tracker.cl:1337/announce",
    "udp://tracker.openbittorrent.com:6969/announce",
    "udp://exodus.desync.com:6969/announce",
    "udp://tracker.torrent.eu.org:451/announce",
    "udp://tracker.moeking.me:6969/announce"
]

def load_mapping():
    if os.path.exists(config_file):
        try:
            with open(config_file, 'r') as f: return json.load(f)
        except: return {}
    return {}

def save_mapping(mapping):
    with open(config_file, 'w') as f: json.dump(mapping, f)

payload_dir_map = load_mapping()
active_seeds = set()

# تهيئة جلسة libtorrent (ننشئها مرة واحدة لجميع التورنتات)
lt_session = lt.session()
lt_session.listen_on(6881, 6891)

# === 2. وظيفة السيدنج (باستخدام libtorrent) ===
def start_libtorrent_seed(torrent_path, target_dir, torrent_name):
    print(f"\n🟢 [Seed]: بدأ العمل على: {torrent_name} (التقرير هيتحدث كل 60 ثانية)")

    try:
        # قراءة بيانات التورنت
        info = lt.torrent_info(torrent_path)

        # إعدادات الإضافة (تخطي الفحص وتحديد المسار)
        params = {
            'save_path': target_dir,
            'storage_mode': lt.storage_mode_t.storage_mode_sparse,
            'ti': info
        }

        # إضافة التورنت وبدء العمل
        handle = lt_session.add_torrent(params)

        def log_printer():
            last_print_time = 0
            while True:
                s = handle.status()
                state_str = ['queued', 'checking', 'downloading metadata',
                             'downloading', 'finished', 'seeding', 'allocating', 'checking fastresume']

                current_time = time.time()

                # التايمر: هل عدى 60 ثانية من آخر مرة طبعنا فيها؟
                if current_time - last_print_time >= 60:
                    # حساب السرعة
                    up_speed = s.upload_rate / 1000
                    unit = "KB/s"
                    if up_speed > 1000:
                        up_speed = up_speed / 1000
                        unit = "MB/s"

                    print(f"\n📊 [{torrent_name}] | الحالة: {state_str[s.state]} | "
                          f"المتصلين: {s.num_peers} | سرعة الرفع: {up_speed:.2f} {unit} | "
                          f"مرفوع كلياً: {s.total_payload_upload / (1024*1024):.2f} MB")

                    sys.stdout.flush()
                    last_print_time = current_time

                # نوم لمدة ثانية لتخفيف الضغط على المعالج
                time.sleep(1)

        # تشغيل الطباعة في الخلفية
        threading.Thread(target=log_printer, daemon=True).start()

    except Exception as e:
        print(f"\n⚠️ حدث خطأ أثناء تشغيل {torrent_name}: {e}")

# === 3. المراقب الآلي ===
def watchdog():
    global payload_dir_map
    while True:
        files = glob.glob(f"{seed_folder}/*.torrent")
        for f in files:
            if f not in active_seeds:
                try:
                    ut = Torrent.read(f)
                    if ut.name in payload_dir_map:
                        start_libtorrent_seed(f, payload_dir_map[ut.name], ut.name)
                        active_seeds.add(f)
                except Exception as e:
                    pass
        time.sleep(5)

threading.Thread(target=watchdog, daemon=True).start()

# === 4. واجهة المستخدم (القائمة الرئيسية) ===
def main_menu():
    global payload_dir_map
    print("\n" + "="*50)
    print("🤖 لوحة تحكم MoApp للتورنت (الذاكرة الذكية) 🤖")
    print("="*50)

    while True:
        print("\nاختر العملية التي تريد القيام بها:")
        print("1️⃣ - إنشاء تورنت جديد (Create)")
        print("2️⃣ - تفعيل سيدنج لملف مرفوع (Manual Seed)")
        print("3️⃣ - عرض الذاكرة (المسارات المحفوظة)")
        print("0️⃣ - إنهاء البرنامج")

        choice = input("\n✏️ اختيارك: ").strip()

        if choice == '1':
            target = input("✏️ اسحب المجلد/الملف من الدرايف هنا (Path): ").strip()
            if os.path.exists(target):

                # --- خيار تحديد نوع التورنت داخلي أو خارجي ---
                print("\n⚙️ اختر نوع التورنت:")
                print("1 - داخلي (Private - ArabP2P)")
                print("2 - خارجي (Public - Open Trackers)")
                t_type = input("✏️ نوع التورنت: ").strip()
                # ---------------------------------------------

                base_name = os.path.basename(target.rstrip('/'))

                # تمييز الاسم بناءً على النوع المراد إنشاؤه
                prefix = "[MoApp-Public]" if t_type == '2' else "[MoApp-Private]"
                t_name = f"{prefix}.{base_name}.torrent"
                parent = os.path.dirname(target) or "."

                print("🚀 جاري البناء... (قد يستغرق وقتاً للحجم الكبير)")

                if t_type == '2':
                    # إنشاء تورنت خارجي عام مع التراكرز النشطة
                    t = Torrent(path=target, trackers=public_trackers, private=False)
                    print("🌐 تم إعداد تورنت خارجي (عام).")
                else:
                    # إنشاء تورنت داخلي خاص بالتراكر المعتاد
                    t = Torrent(path=target, trackers=['http://www.arabp2p.net:2052/announce'], private=True)
                    print("🔒 تم إعداد تورنت داخلي (خاص).")

                payload_dir_map[t.name] = parent
                save_mapping(payload_dir_map)

                t.generate()
                t.write(f'/content/{t_name}')
                print(f"✅ تم الإنشاء: {t_name}")
                print("👉 حمله من القائمة الجانبية وارفع للموقع.")
            else: print("❌ المسار غير موجود!")

        elif choice == '2':
            files = glob.glob(f"{seed_folder}/*.torrent")
            if not files:
                print("⚠️ مجلد Seed_Torrents فارغ! ارفع ملفاتك النهائية فيه أولاً.")
                continue

            for f in files:
                try:
                    ut = Torrent.read(f)
                    if ut.name not in payload_dir_map:
                        print(f"\n❓ التورنت [{ut.name}] مساره مجهول.")
                        p_path = input(f"✏️ أدخل المسار 'الأب' لهذا المحتوى على الدرايف: ").strip()
                        if os.path.exists(p_path):
                            payload_dir_map[ut.name] = p_path
                            save_mapping(payload_dir_map)
                            print("✅ تم الحفظ. سيقوم المراقب بلقطه وتشغيله فوراً.")
                        else: print("❌ مسار خطأ، تم التخطي.")
                except: pass

        elif choice == '3':
            print("\n📂 المسارات المحفوظة في الذاكرة:")
            for k, v in payload_dir_map.items(): print(f" - {k} -> {v}")

        elif choice == '0':
            print("👋 مع السلامة!")
            break
        else: print("❌ اختيار غير صحيح.")

if __name__ == "__main__":
    main_menu()
    # إبقاء الكود حياً
    try:
        while True: time.sleep(1)
    except KeyboardInterrupt: pass